# 06. Sorting, Reindexing & Reshaping: Beginner Guide

### 📌 Overview
Master **06. Sorting, Reindexing & Reshaping: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Sorting**: Covers `.sort_values()` and `.sort_index()`.
- **Reindexing**: Covers `.reindex()`, `.reset_index()`, and `.set_index()`.
- **Reshaping & Pivoting**: Covers `pd.melt()`, `df.pivot()`, `df.pivot_table()`, `.stack()`, and `.unstack()`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  transaction_amount card_type  \
0       TX109326      C55082       M3549              607.78      Visa   
1       TX106376      C76616       M3068             1819.11      Visa   

  transaction_status device_type  account_age_months     transaction_date  \
0           Reversed      Mobile                   8  2026-02-17 08:28:57   
1            Pending         POS                  28          03-Jan-2025   

  region  is_fraud  
0  North         0  
1   West         1  


### 🔹 Sorting Values with `.sort_values()`
- **What it does:** Sorts transactions by amount in descending order.
- **Syntax:** `df.sort_values(by='transaction_amount', ascending=False)`
- **Operation:** `sorted_tx = df.sort_values(by='transaction_amount', ascending=False)`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [2]:
sorted_tx = df.sort_values(by='transaction_amount', ascending=False)
print('Top 3 Largest Transactions:\n', sorted_tx[['transaction_id', 'transaction_amount', 'card_type', 'is_fraud']].head(3))

Top 3 Largest Transactions:
       transaction_id  transaction_amount   card_type  is_fraud
5214        TX113073             1999.98  MasterCard         0
11115       TX101707             1999.85        Amex         1
7173        TX108075             1999.74  MasterCard         1


### 🔹 Sorting by Index with `.sort_index()`
- **What it does:** Sorts DataFrame chronologically by date index.
- **Syntax:** `df.set_index('transaction_date').sort_index()`
- **Operation:** `ts_df = df.set_index('transaction_date')`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [3]:
ts_df = df.set_index('transaction_date')
print('Sorted by Index Head:\n', ts_df.head(2))

Sorted by Index Head:
                     transaction_id customer_id merchant_id  \
transaction_date                                             
2026-02-17 08:28:57       TX109326      C55082       M3549   
03-Jan-2025               TX106376      C76616       M3068   

                     transaction_amount card_type transaction_status  \
transaction_date                                                       
2026-02-17 08:28:57              607.78      Visa           Reversed   
03-Jan-2025                     1819.11      Visa            Pending   

                    device_type  account_age_months region  is_fraud  
transaction_date                                                      
2026-02-17 08:28:57      Mobile                   8  North         0  
03-Jan-2025                 POS                  28   West         1  


### 🔹 Conforming to New Labels: `.reindex()`
- **What it does:** Reindexes DataFrame with a custom index range.
- **Syntax:** `df.reindex(new_index, fill_value=0)`
- **Operation:** `reindexed_tx = df.head(5).reindex([0, 1, 2, 999], fill_value=0.0)`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [4]:
reindexed_tx = df.head(5).reindex([0, 1, 2, 999], fill_value=0.0)
print('Reindexed Transaction Slice:\n', reindexed_tx[['transaction_id', 'transaction_amount']])

Reindexed Transaction Slice:
     transaction_id  transaction_amount
0         TX109326              607.78
1         TX106376             1819.11
2         TX103301               64.08
999            0.0                0.00


### 🔹 Resetting Index: `.reset_index()`
- **What it does:** Restores default 0-indexed integer RangeIndex.
- **Syntax:** `sorted_tx.reset_index(drop=True)`
- **Key Note:** Sets only store unique elements and provide $O(1)$ instant lookup time, making `item in my_set` extremely fast.

In [5]:
print('Reset Index Head:\n', sorted_tx.reset_index(drop=True)[['transaction_id', 'transaction_amount']].head(3))

Reset Index Head:
   transaction_id  transaction_amount
0       TX113073             1999.98
1       TX101707             1999.85
2       TX108075             1999.74


### 🔹 Setting Index: `.set_index()`
- **What it does:** Designates `transaction_id` as primary index.
- **Syntax:** `df.set_index('transaction_id')`
- **Operation:** `tx_indexed = df.set_index('transaction_id')`
- **Key Note:** Sets only store unique elements and provide $O(1)$ instant lookup time, making `item in my_set` extremely fast.

In [6]:
tx_indexed = df.set_index('transaction_id')
print('Transaction ID Indexed Head:\n', tx_indexed.head(2))

Transaction ID Indexed Head:
                customer_id merchant_id  transaction_amount card_type  \
transaction_id                                                         
TX109326            C55082       M3549              607.78      Visa   
TX106376            C76616       M3068             1819.11      Visa   

               transaction_status device_type  account_age_months  \
transaction_id                                                      
TX109326                 Reversed      Mobile                   8   
TX106376                  Pending         POS                  28   

                   transaction_date region  is_fraud  
transaction_id                                        
TX109326        2026-02-17 08:28:57  North         0  
TX106376                03-Jan-2025   West         1  


### 🔹 Unpivoting Wide to Long: `pd.melt()`
- **What it does:** Unpivots metric columns into long format key-value pairs.
- **Syntax:** `pd.melt(df, id_vars=['transaction_id'], value_vars=['transaction_amount', 'account_age_months'])`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [7]:
melted_tx = pd.melt(df.head(5), id_vars=['transaction_id'], value_vars=['transaction_amount', 'account_age_months'], var_name='metric', value_name='metric_value')
print('Melted Transaction Metrics:\n', melted_tx)

Melted Transaction Metrics:
   transaction_id              metric  metric_value
0       TX109326  transaction_amount        607.78
1       TX106376  transaction_amount       1819.11
2       TX103301  transaction_amount         64.08
3       TX110701  transaction_amount       1025.73
4       TX103284  transaction_amount        772.74
5       TX109326  account_age_months          8.00
6       TX106376  account_age_months         28.00
7       TX103301  account_age_months         91.00
8       TX110701  account_age_months         50.00
9       TX103284  account_age_months          5.00


### 🔹 Long to Wide Pivoting: `df.pivot()`
- **What it does:** Pivots unique key combinations into wide format.
- **Syntax:** `df.pivot(index='customer_id', columns='card_type', values='amount')`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [8]:
print('df.pivot Syntax: df.pivot(index="customer_id", columns="card_type", values="amount")')

df.pivot Syntax: df.pivot(index="customer_id", columns="card_type", values="amount")


### 🔹 Pivot Tables: `df.pivot_table()`
- **What it does:** Constructs multidimensional regional card spending matrices with margins.
- **Syntax:** `df.pivot_table(index='region', columns='card_type', values='transaction_amount', aggfunc='mean', margins=True)`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [9]:
piv_spending = df.pivot_table(index='region', columns='card_type', values='transaction_amount', aggfunc='mean', margins=True)
print('Regional Card Spending Matrix (Mean Amount):\n', piv_spending.round(2))

Regional Card Spending Matrix (Mean Amount):
 card_type     Amex  Discover  MasterCard     Visa      All
region                                                    
 East      1052.77    832.92     1147.77   841.70   975.55
 North     1006.64   1169.58     1164.63   914.34  1078.73
 South     1041.77   1047.69     1126.32  1222.24  1096.62
 West       818.63   1068.40      938.36   808.63   899.14
East       1000.96   1006.02     1001.76   969.95   994.70
North       986.13   1014.09      994.79  1013.21  1002.27
South       987.27   1017.76     1008.33  1011.52  1005.98
West       1029.77   1016.82      986.61  1002.68  1009.07
east       1055.34   1153.96     1254.71   954.76  1111.78
north      1030.89    946.74     1033.94  1321.96  1100.16
south      1126.37   1037.64     1171.81  1014.28  1089.79
west       1216.81   1301.80      909.85  1054.59  1119.38
All        1002.94   1016.84     1002.26   999.25  1005.33


### 🔹 Stacking Columns to Rows: `.stack()`
- **What it does:** Stacks pivoted card columns into hierarchical multi-level row index.
- **Syntax:** `piv_spending.stack()`
- **Operation:** `stacked_piv = piv_spending.stack()`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [10]:
stacked_piv = piv_spending.stack()
print('Stacked Pivot Series Head:\n', stacked_piv.head())

Stacked Pivot Series Head:
 region  card_type 
East    Amex          1052.769444
        Discover       832.917059
        MasterCard    1147.773000
        Visa           841.696667
        All            975.553973
dtype: float64


### 🔹 Unstacking Rows to Columns: `.unstack()`
- **What it does:** Unstacks hierarchical row levels back to column headers.
- **Syntax:** `stacked_piv.unstack()`
- **Operation:** `print('Unstacked DataFrame:\n', stacked_piv.unstack().head(2))`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [11]:
print('Unstacked DataFrame:\n', stacked_piv.unstack().head(2))

Unstacked DataFrame:
 card_type         Amex     Discover   MasterCard        Visa          All
region                                                                   
East       1052.769444   832.917059  1147.773000  841.696667   975.553973
North      1006.639231  1169.578800  1164.634118  914.338235  1078.726528


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Regional Fraud Risk Contingency Pivot Table
- **Objective:** Q1: Regional Fraud Risk Contingency Pivot Table
- **Approach:** Generate a pivot contingency matrix showing fraud rates across regions and device types.
- **Syntax:** `df.pivot_table(index='region', columns='device_type', values='is_fraud', aggfunc='mean') * 100`

In [12]:
fraud_matrix = df.pivot_table(index='region', columns='device_type', values='is_fraud', aggfunc='mean') * 100
print('Regional Device Fraud Rate Matrix (%):\n', fraud_matrix.round(2))

Regional Device Fraud Rate Matrix (%):
 device_type    ATM  Desktop  Mobile    POS
region                                    
 East         8.70     9.09    0.00  14.29
 North        0.00     4.76   18.75  10.53
 South       11.11    23.81   16.67   4.55
 West         6.25     0.00    0.00  13.04
East          7.67     9.75   12.18   9.83
North        11.40     9.93   10.53  13.76
South        11.70    10.11   10.90   9.42
West         14.49     9.24    8.75  12.38
east         11.54    25.00   15.00  14.29
north        13.33    18.18   20.00  27.27
south         8.33     7.41   12.50  12.50
west         10.53    23.53    4.35  11.76
